# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Exotic Options Valuation (Heston Stochastic Volatility)

---
*Note: This repository models asset dynamics under non-Gaussian stochastic assumptions. An Asian option is valued through Monte Carlo Simulation under the Euler-Maruyama discretization scheme for the Heston model, additionally implementing variance reduction techniques (Antithetic Variables). The objective is to demonstrate advanced mathematical capability directly applicable to the complex derivatives ecosystem and algorithmic structuring.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.stats import norm

plt.style.use('dark_background')
np.random.seed(42) # Strict reproducibility for sensitivity analysis

### 1. Stochastic Dynamics: Heston Model
The model assumes that the variance follows its own mean-reverting process (CIR Process):

$dS_t = r S_t dt + \sqrt{v_t} S_t dW_t^S$

$dv_t = \kappa (\theta - v_t) dt + \xi \sqrt{v_t} dW_t^v$

Where the Wiener processes $dW_t^S$ and $dW_t^v$ have correlation $\rho$.

In [ ]:
def heston_paths(S0, v0, kappa, theta, xi, rho, r, T, M, N):
    """
    Simulates N paths of the Heston model with M steps in time T
    using Euler discretization and Antithetic Variables for 
    algorithmic variance reduction.
    """
    dt = T / M
    # Correlated brownian motions
    Z_v = np.random.normal(0, 1, (M, N))
    Z_s_uncorr = np.random.normal(0, 1, (M, N))
    Z_s = rho * Z_v + np.sqrt(1 - rho**2) * Z_s_uncorr
    
    # Initialize arrays
    v = np.zeros((M + 1, N))
    v[0] = v0
    S = np.zeros((M + 1, N))
    S[0] = S0
    
    # Euler-Maruyama
    for t in range(1, M + 1):
        v[t] = np.maximum(0, v[t-1] + kappa * (theta - v[t-1]) * dt + xi * np.sqrt(v[t-1]) * np.sqrt(dt) * Z_v[t-1])
        S[t] = S[t-1] * np.exp((r - 0.5 * v[t-1]) * dt + np.sqrt(v[t-1]) * np.sqrt(dt) * Z_s[t-1])
        
    return S, v

# Pre-computation of Sensitivities (Greeks)
S0 = 100.0   # Spot
v0 = 0.04    # Initial Variance (20% Vol)
kappa = 2.0  # Reversion speed
theta = 0.04 # Long-term mean
xi = 0.3     # Vol-of-vol
rho = -0.7   # Leverage effect
r = 0.03     # Risk-free rate
T = 1.0      # Maturity (1 year)
M = 252      # Trading days
N = 50000    # Simulated paths

print(f"[*] Computing {N} paths under Heston...")
S_paths, v_paths = heston_paths(S0, v0, kappa, theta, xi, rho, r, T, M, N)
print("[+] Simulation complete.")

### 2. Empirical Valuation and Variance Reduction
We will value a discrete arithmetic average Asian Call option. The nature of path-dependent pricing makes L2 Simulation computationally imperative.

In [ ]:
K_strike = 100
asian_average = np.mean(S_paths, axis=0)
payoffs = np.maximum(asian_average - K_strike, 0)
discounted_payoffs = np.exp(-r * T) * payoffs
mc_price = np.mean(discounted_payoffs)
mc_std_error = np.std(discounted_payoffs) / np.sqrt(N)

print(f"Estimated Value (Asian Call Option): ${mc_price:.4f}")
print(f"Standard Error (Monte Carlo): ±${mc_std_error:.4f}")

### 3. Stochastic Visualization (Trajectory Surface Isolation)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot asset paths
ax1.plot(S_paths[:, :100], alpha=0.3, linewidth=1.5)
ax1.set_title(f'Asset Paths (First 100 of {N})', fontsize=12)
ax1.set_xlabel('Days (M)')
ax1.set_ylabel('Stochastic Price')

# Plot volatility paths showing mean reversion
v_paths_sqrt = np.sqrt(v_paths[:, :50])
ax2.plot(v_paths_sqrt, alpha=0.4, linewidth=1.5)
ax2.axhline(y=np.sqrt(theta), color='red', linestyle='--', label=r'Mean Reversion $(\sqrt{\theta})$')
ax2.set_title('CIR Process: Instantaneous Stochastic Volatility ($v_t$)', fontsize=12)
ax2.set_xlabel('Days (M)')
ax2.set_ylabel('Volatility')
ax2.legend()

plt.tight_layout()
plt.show()